# **1. Perkenalan Dataset**

Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.

**Dataset: Red Chili Pepper Pests Dataset**

- **Sumber:** Kaggle (https://www.kaggle.com/datasets/indraagustian/red-chili-pepper-pests-dataset)
- **Jenis:** Image Classification (Unstructured Data)
- **Jumlah Kelas:** 4 kelas hama cabai merah:
  - MP (kutu daun/aphid) - Class 0
  - BT (kutu kebul/whitefly) - Class 1
  - T (thrips) - Class 2
  - C (ulat/caterpillar) - Class 3
- **Format:** Gambar JPEG dengan label YOLO-style (.txt files)
- **Identifikasi kelas:** Berdasarkan prefix filename (kutu-daun, kutu-kebul, thrips, ulat)

# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import os
import glob
import random

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight
from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [ ]:
DATASET_DIR = os.environ.get('DATASET_DIR', '/kaggle/input/datasets/indraagustian/red-chili-pepper-pests-dataset')

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

CLASS_NAMES = ['MP (kutu daun)', 'BT (kutu kebul)', 'T (thrips)', 'C (ulat)']
NUM_CLASSES = len(CLASS_NAMES)

CLASS_MAP = {
    'kutu-daun': 0,
    'kutu-kebul': 1,
    'thrips': 2,
    'thrips-baru': 2,
    'ulat': 3,
}


def get_class_from_filename(filename):
    basename = os.path.basename(filename)
    prefix = basename.split('--')[0].lower()

    if prefix in CLASS_MAP:
        return CLASS_MAP[prefix]

    label_path = filename.rsplit('.', 1)[0] + '.txt'
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            first_line = f.readline().strip()
            if first_line:
                return int(first_line.split()[0])

    raise ValueError(f"Tidak bisa menentukan kelas untuk: {filename}")


def collect_images_and_labels(image_dir):
    image_files = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))
    labels = []
    valid_files = []
    for img_path in image_files:
        try:
            label = get_class_from_filename(img_path)
            labels.append(label)
            valid_files.append(img_path)
        except ValueError as e:
            print(f"Peringatan: {e}")

    return valid_files, labels

In [ ]:
print("Mengumpulkan data training")
train_files, train_labels = collect_images_and_labels(
    os.path.join(DATASET_DIR, 'train', 'images')
)
print(f"Training: {len(train_files)} gambar")

In [ ]:
print("Mengumpulkan data validasi")
val_files_1, val_labels_1 = collect_images_and_labels(
    os.path.join(DATASET_DIR, 'val', 'images')
)
val_files_2, val_labels_2 = collect_images_and_labels(
    os.path.join(DATASET_DIR, 'val', 'valid', 'images')
)
val_files = val_files_1 + val_files_2
val_labels = val_labels_1 + val_labels_2
print(f"Validasi: {len(val_files)} gabungan ({len(val_files_1)} + {len(val_files_2)})")

In [ ]:
print("Mengumpulkan data test")
test_files, test_labels = collect_images_and_labels(
    os.path.join(DATASET_DIR, 'test', 'images')
)
print(f"Test: {len(test_files)} gambar")

for split_name, labels in [('Train', train_labels), ('Val', val_labels), ('Test', test_labels)]:
    unique, counts = np.unique(labels, return_counts=True)
    print(f"\nDistribusi {split_name}:")
    for cls_id, count in zip(unique, counts):
        print(f"  {CLASS_NAMES[cls_id]}: {count} ({count/len(labels)*100:.1f}%)")

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (split_name, labels) in zip(axes, [('Train', train_labels), ('Val', val_labels), ('Test', test_labels)]):
    unique, counts = np.unique(labels, return_counts=True)
    bars = ax.bar([CLASS_NAMES[i] for i in unique], counts, color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'])
    ax.set_title(f'Distribusi Kelas - {split_name}', fontsize=13)
    ax.set_ylabel('Jumlah Gambar')
    ax.tick_params(axis='x', rotation=15)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                str(count), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sample_per_class = {}
for f, l in zip(train_files, train_labels):
    if l not in sample_per_class:
        sample_per_class[l] = f
    if len(sample_per_class) == NUM_CLASSES:
        break

fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(16, 4))
for idx in range(NUM_CLASSES):
    img = tf.io.read_file(sample_per_class[idx])
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    axes[idx].imshow(img.numpy())
    axes[idx].set_title(CLASS_NAMES[idx], fontsize=11)
    axes[idx].axis('off')

plt.suptitle('Contoh Gambar per Kelas', fontsize=14)
plt.tight_layout()
plt.savefig('eda_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
total_images = len(train_files) + len(val_files) + len(test_files)
print(f"Total gambar: {total_images}")
print(f"  Train: {len(train_files)} ({len(train_files)/total_images*100:.1f}%)")
print(f"  Val:   {len(val_files)} ({len(val_files)/total_images*100:.1f}%)")
print(f"  Test:  {len(test_files)} ({len(test_files)/total_images*100:.1f}%)")
print(f"\nImage size target: {IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Jumlah kelas: {NUM_CLASSES}")

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
], name="data_augmentation")


def parse_image(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    label = tf.one_hot(label, depth=NUM_CLASSES)
    return img, label


def augment_image(image, label):
    image = data_augmentation(image)
    return image, label


def create_dataset(file_paths, labels, batch_size=BATCH_SIZE, shuffle=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(file_paths), seed=42)
    ds = ds.map(parse_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size)
    if augment:
        ds = ds.map(augment_image, num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

In [ ]:
all_train_files = train_files + val_files
all_train_labels = train_labels + val_labels

train_files_final, val_files_final, train_labels_final, val_labels_final = train_test_split(
    all_train_files,
    all_train_labels,
    test_size=0.15,
    stratify=all_train_labels,
    random_state=42
)

print(f"Train : {len(train_files_final)} gambar")
print(f"Val   : {len(val_files_final)} gambar")
print(f"Test  : {len(test_files)} gambar")

In [ ]:
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.array([0, 1, 2, 3]),
    y=np.array(train_labels_final)
)
class_weight_dict = dict(enumerate(class_weights))
print(f"\nClass weights:")
for i, w in class_weight_dict.items():
    print(f"  {CLASS_NAMES[i]}: weight={w:.4f}")

In [ ]:
train_ds = create_dataset(train_files_final, train_labels_final, shuffle=True, augment=True)
val_ds = create_dataset(val_files_final, val_labels_final, shuffle=False, augment=False)
test_ds = create_dataset(test_files, test_labels, shuffle=False, augment=False)

print("\nDataset siap untuk training:")
print(f"  Train batches: {len(list(train_ds))}")
print(f"  Val batches:   {len(list(val_ds))}")
print(f"  Test batches:  {len(list(test_ds))}")

In [ ]:
plt.figure(figsize=(12, 8))
sample_images, sample_labels = next(iter(train_ds))
for i in range(min(16, len(sample_images))):
    plt.subplot(4, 4, i + 1)
    plt.imshow(sample_images[i])
    label_idx = tf.argmax(sample_labels[i]).numpy()
    plt.title(CLASS_NAMES[label_idx], fontsize=9)
    plt.axis('off')
plt.suptitle('Contoh Gambar Training (dengan Augmentation)', fontsize=14)
plt.tight_layout()
plt.savefig('preprocessing_augmented_samples.png', dpi=150, bbox_inches='tight')
plt.show()